# Domain Code Evaluation Benchmark Generator

Generate **execution-verified** coding benchmarks for any domain using SDG Hub's
code interpreter.

**The pipeline** (inspired by [AutoCodeBench](https://arxiv.org/abs/2508.09101)
and [CRUXEval](https://arxiv.org/abs/2401.03065)):

```
domain + function specs (seed)
  --> LLM generates function implementation        (Phase 1)
  --> LLM generates assert-based test suite        (Phase 2)
  --> PythonInterpreterBlock verifies execution     (Phase 3)
  --> LLM reverse-generates problem description     (Phase 4)
  --> verified benchmark: (problem, solution, tests)
```

**Why this matters:** The problem description is generated *last*, from verified
code -- not the other way around. Every benchmark problem is backed by a solution
and test suite that provably executes. No hallucinated test cases, no broken
reference solutions.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

# pip install sdg_hub[code]        # pydantic-monty for sandboxed execution
# pip install sdg_hub[examples]    # notebook dependencies

In [ ]:
import warnings

import nest_asyncio
import pandas as pd

from sdg_hub import Flow, FlowRegistry

warnings.filterwarnings("ignore")
nest_asyncio.apply()

## 2. Load and Inspect the Flow

In [ ]:
FlowRegistry.discover_flows()

flow = Flow.from_yaml(FlowRegistry.get_flow_path("domain-code-eval"))
flow.print_info()

## 3. Configure the Model

In [ ]:
flow.set_model_config(
    model="gpt-5.1-codex-mini",
)

# Alternatives:
# flow.set_model_config(model="gpt-4o", api_key="...")
# flow.set_model_config(
#     model="hosted_vllm/Qwen/Qwen2.5-Coder-32B-Instruct",
#     api_base="http://localhost:8101/v1",
#     api_key="EMPTY",
# )

print("Model configured")

## 4. Prepare Seed Data

100 function specifications across 6 domains. Each row describes *what* the function
should do -- the LLM generates the implementation, tests, and problem description.

In [ ]:
# Each entry is (function_spec, time_complexity)
specs = {
    "financial calculations": {
        "beginner": [
            ("simple interest calculator given principal, rate, and time", "O(1)"),
            ("percentage change between two values", "O(1)"),
            ("discount price given original price and discount percentage", "O(1)"),
            ("tip calculator that splits bill among people", "O(1)"),
        ],
        "intermediate": [
            ("compound interest with variable compounding periods", "O(1)"),
            ("loan monthly payment calculator using amortization formula", "O(1)"),
            ("tax bracket calculator with progressive brackets given as list of (threshold, rate) tuples", "O(n)"),
            ("weighted average calculator given values and weights", "O(n)"),
            ("moving average of a list of prices with configurable window size", "O(n)"),
            ("present value of future cash flows with given discount rate", "O(n)"),
            ("profit margin calculator (gross and net) from revenue and costs", "O(1)"),
        ],
        "advanced": [
            ("bond price calculator from coupon rate, yield, face value, and periods", "O(n)"),
            ("internal rate of return (IRR) approximation using bisection method", "O(n log(1/eps))"),
            ("depreciation schedule using declining balance method", "O(n)"),
            ("currency converter with cross-rate calculation through a base currency", "O(n)"),
        ],
    },
    "text processing": {
        "beginner": [
            ("word frequency counter returning dict of word counts", "O(n)"),
            ("title case converter that keeps small words (a, an, the, of) lowercase", "O(n)"),
            ("vowel and consonant counter returning a dict with counts", "O(n)"),
            ("string truncator that adds ellipsis if string exceeds max length", "O(1)"),
        ],
        "intermediate": [
            ("run-length encoder that compresses 'aaabbc' to '3a2b1c'", "O(n)"),
            ("run-length decoder that expands '3a2b1c' to 'aaabbc'", "O(n)"),
            ("Caesar cipher encoder/decoder with configurable shift", "O(n)"),
            ("bracket validator that checks matching parentheses, brackets, and braces", "O(n)"),
            ("slug generator that converts titles to URL-safe lowercase strings", "O(n)"),
            ("sentence splitter that handles abbreviations like 'Dr.' and 'U.S.A.'", "O(n)"),
            ("text wrapper that breaks text into lines of max width without splitting words", "O(n)"),
        ],
        "advanced": [
            ("regex-free pattern matcher supporting * (any chars) and ? (single char) wildcards", "O(n*m)"),
            ("Levenshtein edit distance calculator between two strings", "O(n*m)"),
            ("longest common subsequence of two strings", "O(n*m)"),
            ("template string renderer that replaces {{variable}} placeholders from a dict", "O(n*k)"),
        ],
    },
    "data transformation": {
        "beginner": [
            ("flatten a nested list of arbitrary depth into a single flat list", "O(n)"),
            ("deduplicate a list while preserving original order", "O(n)"),
            ("chunk a list into groups of n elements", "O(n)"),
            ("zip two dictionaries by shared keys into a dict of tuples", "O(n)"),
        ],
        "intermediate": [
            ("group a list of dicts by a given key field", "O(n)"),
            ("pivot a list of records: given rows with (category, metric, value), produce {category: {metric: value}}", "O(n)"),
            ("transpose a matrix (list of lists)", "O(n*m)"),
            ("merge two sorted lists into one sorted list", "O(n+m)"),
            ("invert a dictionary (swap keys and values, handling duplicate values as lists)", "O(n)"),
            ("running sum of a list of numbers", "O(n)"),
            ("interleave two lists element by element, handling unequal lengths", "O(n+m)"),
        ],
        "advanced": [
            ("deep merge two nested dicts recursively (lists concatenate, dicts merge, scalars overwrite)", "O(n)"),
            ("topological sort of a dependency graph given as adjacency dict", "O(V+E)"),
            ("JSON path extractor: get nested value from dict using dot notation like 'a.b.0.c'", "O(d)"),
            ("diff two flat dicts returning added, removed, and changed keys with old/new values", "O(n)"),
        ],
    },
    "mathematical functions": {
        "beginner": [
            ("greatest common divisor (GCD) of two numbers using Euclidean algorithm", "O(log(min(a,b)))"),
            ("check if a number is prime", "O(sqrt(n))"),
            ("factorial of a non-negative integer", "O(n)"),
            ("Fibonacci number at position n (iterative)", "O(n)"),
        ],
        "intermediate": [
            ("prime factorization returning list of (prime, exponent) tuples", "O(sqrt(n))"),
            ("combination calculator nCr without using factorial for large numbers", "O(min(r, n-r))"),
            ("mean, median, and mode of a list of numbers returned as a dict", "O(n log n)"),
            ("standard deviation of a list of numbers (population)", "O(n)"),
            ("Euclidean distance between two points in n-dimensional space", "O(n)"),
            ("base converter from decimal to any base (2-36) returning string", "O(log n)"),
            ("matrix multiplication for two 2D matrices (lists of lists)", "O(n^3)"),
        ],
        "advanced": [
            ("Newton's method for finding square root to specified precision", "O(log(1/eps))"),
            ("polynomial evaluator from list of coefficients using Horner's method", "O(n)"),
            ("numerical integration using Simpson's rule", "O(n)"),
            ("solve a system of 2 linear equations (2x2) returning (x, y) or None if no solution", "O(1)"),
        ],
    },
    "algorithms": {
        "beginner": [
            ("binary search returning index or -1 if not found", "O(log n)"),
            ("bubble sort implementation", "O(n^2)"),
            ("selection sort implementation", "O(n^2)"),
            ("reverse a singly linked list represented as list of (value, next_index) tuples", "O(n)"),
        ],
        "intermediate": [
            ("merge sort implementation", "O(n log n)"),
            ("stack implementation using a list with push, pop, peek, is_empty methods (as a class)", "O(1) per operation"),
            ("queue implementation using a list with enqueue, dequeue, peek, is_empty methods (as a class)", "O(1) per operation"),
            ("breadth-first traversal of a tree represented as adjacency dict returning list of values", "O(V+E)"),
            ("depth-first traversal (pre-order) of a tree represented as adjacency dict", "O(V+E)"),
            ("binary search tree: insert, search, and in-order traversal (as a class)", "O(n) worst case"),
            ("0/1 knapsack problem returning maximum value given weights, values, and capacity", "O(n*W)"),
        ],
        "advanced": [
            ("Dijkstra's shortest path algorithm on weighted adjacency dict", "O(V^2)"),
            ("longest increasing subsequence length", "O(n log n)"),
            ("minimum number of coins for a given amount (coin change problem)", "O(n*amount)"),
            ("detect cycle in a directed graph given as adjacency dict", "O(V+E)"),
        ],
    },
    "string utilities": {
        "beginner": [
            ("palindrome checker (ignoring case and non-alphanumeric characters)", "O(n)"),
            ("anagram checker for two strings", "O(n)"),
            ("count occurrences of each character returning sorted dict", "O(n log n)"),
            ("camelCase to snake_case converter", "O(n)"),
        ],
        "intermediate": [
            ("snake_case to camelCase converter", "O(n)"),
            ("version string comparator: return -1, 0, or 1 for two version strings like '1.2.3'", "O(n)"),
            ("Roman numeral to integer converter", "O(n)"),
            ("integer to Roman numeral converter", "O(1)"),
            ("IP address validator (IPv4) returning True/False", "O(1)"),
            ("credit card number masker showing only last 4 digits", "O(n)"),
        ],
        "advanced": [
            ("URL query string parser returning nested dict from 'a=1&b=2&c[0]=x&c[1]=y'", "O(n)"),
            ("simple expression evaluator for +, -, *, / with parentheses (no eval)", "O(n)"),
            ("password strength scorer (0-100) based on length, diversity, patterns", "O(n)"),
        ],
    },
}

# Flatten into a DataFrame
rows = []
for domain, by_difficulty in specs.items():
    for difficulty, func_specs in by_difficulty.items():
        for spec, complexity in func_specs:
            rows.append({
                "domain": domain,
                "function_spec": spec,
                "difficulty": difficulty,
                "time_complexity": complexity,
            })

seed_data = pd.DataFrame(rows)

print(f"Seed dataset: {len(seed_data)} specs across {seed_data['domain'].nunique()} domains")
print(f"\nComplexity classes: {sorted(seed_data['time_complexity'].unique())}")
print("\nDistribution by domain:")
print(seed_data["domain"].value_counts().to_string())
print("\nDistribution by difficulty:")
print(seed_data["difficulty"].value_counts().to_string())

## 5. Generate the Benchmark

The flow makes 3 LLM calls per row (function, tests, problem description) plus
1 execution call. For 100 rows this takes a few minutes with async.

In [ ]:
results = flow.generate(seed_data)

print(f"Generated {len(results)} benchmark problems")
print(f"Columns: {list(results.columns)}")

## 6. Analyze Results

In [ ]:
# Execution verification results
results["verified"] = results["execution_result"].apply(lambda r: r.get("success", False))

passed = results["verified"].sum()
total = len(results)
print(f"Verification: {passed}/{total} passed ({passed/total*100:.1f}%)\n")

# By domain
print("Pass rate by domain:")
for domain, group in results.groupby("domain"):
    p = group["verified"].sum()
    t = len(group)
    print(f"  {domain:25s}  {p:2d}/{t:2d}  ({p/t*100:.0f}%)")

# By difficulty
print("\nPass rate by difficulty:")
for diff, group in results.groupby("difficulty"):
    p = group["verified"].sum()
    t = len(group)
    print(f"  {diff:15s}  {p:2d}/{t:2d}  ({p/t*100:.0f}%)")

In [ ]:
# Example: a verified benchmark problem
verified = results[results["verified"]]
if len(verified) > 0:
    ex = verified.iloc[0]
    print(f"Domain: {ex['domain']} | Difficulty: {ex['difficulty']}")
    print(f"Spec: {ex['function_spec']}\n")
    print("=" * 70)
    print("PROBLEM DESCRIPTION (what the model sees):")
    print("=" * 70)
    print(ex["problem_description"])
    print()
    print("=" * 70)
    print("REFERENCE SOLUTION (hidden from the model):")
    print("=" * 70)
    print(ex["function_code"])
    print()
    print("=" * 70)
    print("TEST SUITE (used to grade the model's output):")
    print("=" * 70)
    print(ex["test_code"][:500], "..." if len(ex["test_code"]) > 500 else "")

In [ ]:
# Example: a failed problem (shows what the interpreter caught)
failed = results[~results["verified"]]
if len(failed) > 0:
    ex = failed.iloc[0]
    print(f"Domain: {ex['domain']} | Difficulty: {ex['difficulty']}")
    print(f"Spec: {ex['function_spec']}\n")
    print("Error caught by interpreter:")
    print(ex["execution_result"].get("error", "unknown")[:300])
else:
    print("All problems passed verification!")

## 7. Export Verified Benchmark

Filter to keep only execution-verified problems. The final benchmark contains:
- `problem_description` -- what you show the model being evaluated
- `function_code` -- the reference solution (for comparison)
- `test_code` -- the test suite to grade the model's output

In [ ]:
benchmark_cols = [
    "domain", "difficulty", "time_complexity", "function_spec",
    "problem_description", "function_code", "test_code", "input_generator",
]
# Only include columns that exist (input_generator may be missing if tag wasn't parsed)
available_cols = [c for c in benchmark_cols if c in results.columns]

benchmark = results[results["verified"]][available_cols].reset_index(drop=True)

print(f"Final benchmark: {len(benchmark)} verified problems (from {len(results)} generated)")
print(f"Domains: {sorted(benchmark['domain'].unique())}")
print(f"Complexity classes: {sorted(benchmark['time_complexity'].unique())}")

benchmark

In [ ]:
# Save the benchmark
# benchmark.to_json("domain_code_benchmark.jsonl", orient="records", lines=True)
# benchmark.to_parquet("domain_code_benchmark.parquet")

# Or push to HuggingFace Hub:
# from datasets import Dataset
# Dataset.from_pandas(benchmark).push_to_hub("your-org/domain-code-eval")

## 8. Evaluate a Coding Model

Use the benchmark to score a model. Give it the `problem_description`, have it
generate code, then run the `test_code` against it.

In [ ]:
from sdg_hub.core.blocks.code import PythonInterpreterBlock

# Set up the interpreter for grading
grader = PythonInterpreterBlock(
    block_name="grader",
    input_cols=["candidate_code"],
    output_cols=["grade_result"],
    interpreter_framework="monty",
    timeout=10.0,
)

# Pick a sample of 5 problems to evaluate
sample = benchmark.head(5).copy()

# Simulate a model's output: use the reference solution as if the model generated it
# In practice, you'd call your model here:
#   model_output = your_model.generate(row["problem_description"])
sample["candidate_code"] = sample.apply(
    lambda row: row["function_code"] + "\n\n" + row["test_code"],
    axis=1,
)

graded = grader(sample)

for _, row in graded.iterrows():
    status = "PASS" if row["grade_result"]["success"] else "FAIL"
    print(f"[{status}] {row['function_spec'][:60]}")

model_pass_rate = graded["grade_result"].apply(lambda r: r["success"]).mean()
print(f"\nModel pass@1: {model_pass_rate*100:.0f}%")

## Next Steps

- **Add your domain:** Replace the seed data with function specs from your team's
  domain (ML pipelines, DevOps scripts, bioinformatics, etc.)
- **Scale up:** Generate 500+ problems and filter to build a robust benchmark
- **Difficulty control:** Run a weaker model against the benchmark, discard problems
  it solves 10/10 times (following AutoCodeBench's filtering)
- **Multi-language:** Extend to other languages by swapping the interpreter backend
- **Evaluate models:** Use the benchmark to compare coding models on your domain